In [16]:
import pandas as pd
import numpy as np

## Transformer
#### Layer-wise performances

In [17]:
N_LAYERS = [3, 5, 7, 9, 11, 13, 15, 17, 20]
dfs = []
for layer in N_LAYERS:
    df = pd.read_csv(f"Results/test_metrics/{layer}layers/test_metrics_{layer}layers.csv")
    df = df.drop(columns=["output"])
    df.insert(0, "num_layers", [layer])
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

In [18]:
df_all

,num_layers,R2,RMSE,MAE
0,3,0.698622,7.487138,5.833248
1,5,0.698010,7.494740,5.841167
2,7,0.674122,7.785518,5.975801
3,9,0.688740,7.608894,5.989590
4,11,0.698269,7.491516,5.855290
5,13,0.690058,7.592772,5.965261
6,15,0.683662,7.670712,5.924325
7,17,0.696140,7.517903,5.966175
8,20,0.692473,7.563128,5.924075


#### Model reduction over 20 layers

In [19]:
df = pd.read_csv("Results/test_metrics/20layers/test_metrics_20layers.csv")
df.insert(0, "experiment", [""])
df[["R2_drop", "RMSE_drop", "MAE_drop"]] = ""
df = df.drop(columns=["output"])

separator_str = "=== 72 features: 100% ==="
separator = pd.DataFrame([["", separator_str, "", "", "", "", ""]], columns=df.columns)
df_list = [separator, df]

In [20]:
def compute_drop(df1, df2):

    R2_drop = round(100*(df1.loc[0, "R2"]-df2.loc[0, "R2"]) / df1.loc[0, "R2"], 2)
    RMSE_drop = round(100*(df2.loc[0, "RMSE"]-df1.loc[0, "RMSE"]) / df1.loc[0, "RMSE"], 2)
    MAE_drop = round(100*(df2.loc[0, "MAE"]-df1.loc[0, "MAE"]) / df1.loc[0, "MAE"], 2)
        
    return [R2_drop, RMSE_drop, MAE_drop]

In [21]:
topks = [18, 25]
topks_perc = [25, 35]

for topk, topk_perc in zip(topks, topks_perc):
    df_random = pd.read_csv(f"Results/test_metrics/20layers/test_metrics_20layers_random{topk}.csv")
    drop_random = compute_drop(df, df_random)
    drop_random_str = [str(item) + "%" for item in drop_random]
    df_random[["R2_drop", "RMSE_drop", "MAE_drop"]] = drop_random_str

    df_shap = pd.read_csv(f"Results/test_metrics/20layers/test_metrics_20layers_shap{topk}.csv")
    drop_shap = compute_drop(df, df_shap)
    drop_shap_str = [str(item) + "%" for item in drop_shap]
    df_shap[["R2_drop", "RMSE_drop", "MAE_drop"]] = drop_shap_str
    
    df_attn = pd.read_csv(f"Results/test_metrics/20layers/test_metrics_20layers_sliced{topk}.csv")
    drop_attn = compute_drop(df, df_attn)
    drop_attn_str = [str(item) + "%" for item in drop_attn]
    df_attn[["R2_drop", "RMSE_drop", "MAE_drop"]] = drop_attn_str
      
    df_all = pd.concat([df_random, df_shap, df_attn], ignore_index=True)
    df_all.insert(0, "experiment", ["Random MR", "SHAP MR", "Attention MR"])
    df_all = df_all.drop(columns=["output"])

    separator_str = f"=== {topk} features: {topk_perc}% ==="
    separator = pd.DataFrame([["", separator_str, "", "", "", "", ""]], columns=df.columns)
    df_list = df_list + [separator, df_all]

In [22]:
final_df = pd.concat(df_list, ignore_index=True)

In [23]:
final_df

,experiment,R2,RMSE,MAE,R2_drop,RMSE_drop,MAE_drop
0,,=== 72 features: 100% ===,,,,,
1,,0.692473,7.563128,5.924075,,,
2,,=== 18 features: 25% ===,,,,,
3,Random MR,0.486776,9.770424,7.695175,29.7%,29.18%,29.9%
4,SHAP MR,0.63561,8.232714,6.39421,8.21%,8.85%,7.94%
5,Attention MR,0.586823,8.766533,6.929586,15.26%,15.91%,16.97%
6,,=== 25 features: 35% ===,,,,,
7,Random MR,0.593022,8.700529,6.911261,14.36%,15.04%,16.66%
8,SHAP MR,0.668419,7.853343,6.179348,3.47%,3.84%,4.31%
9,Attention MR,0.619558,8.412099,6.487724,10.53%,11.23%,9.51%


## MLP

In [9]:
df_mlp = pd.read_csv("Results_MLP/test_metrics/test_metrics.csv")
df_mlp.insert(0, "experiment", [""])
df_mlp[["R2_drop", "RMSE_drop", "MAE_drop"]] = ""
df_mlp = df_mlp.drop(columns=["output"])

separator_str_mlp = "=== 72 features: 100% ==="
separator_mlp = pd.DataFrame([["", separator_str_mlp, "", "", "", "", ""]], 
                             columns=df_mlp.columns)
df_mlp_list = [separator_mlp, df_mlp]

In [10]:
for topk, topk_perc in zip(topks, topks_perc):
    df_random_mlp = pd.read_csv(f"Results_MLP/test_metrics/test_metrics_random{topk}.csv")
    drop_random_mlp = compute_drop(df_mlp, df_random_mlp)
    drop_random_str_mlp = [str(item) + "%" for item in drop_random_mlp]
    df_random_mlp[["R2_drop", "RMSE_drop", "MAE_drop"]] = drop_random_str_mlp

    df_shap_mlp = pd.read_csv(f"Results_MLP/test_metrics/test_metrics_shap{topk}.csv")
    drop_shap_mlp = compute_drop(df_mlp, df_shap_mlp)
    drop_shap_str_mlp = [str(item) + "%" for item in drop_shap_mlp]
    df_shap_mlp[["R2_drop", "RMSE_drop", "MAE_drop"]] = drop_shap_str_mlp
     
    df_all_mlp = pd.concat([df_random_mlp, df_shap_mlp], ignore_index=True)
    df_all_mlp.insert(0, "experiment", ["Random MR", "SHAP MR"])
    df_all_mlp = df_all_mlp.drop(columns=["output"])

    separator_str_mlp = f"=== {topk} features: {topk_perc}% ==="
    separator_mlp = pd.DataFrame([["", separator_str_mlp, "", "", "", "", ""]], 
                                 columns=df_mlp.columns)
    df_mlp_list = df_mlp_list + [separator_mlp, df_all_mlp]

In [11]:
final_df_mlp = pd.concat(df_mlp_list, ignore_index=True)

In [12]:
final_df_mlp

,experiment,R2,RMSE,MAE,R2_drop,RMSE_drop,MAE_drop
0,,=== 72 features: 100% ===,,,,,
1,,0.692725,7.560029,6.008455,,,
2,,=== 18 features: 25% ===,,,,,
3,Random MR,0.458852,10.032704,8.151547,33.76%,32.71%,35.67%
4,SHAP MR,0.615088,8.461369,6.635888,11.21%,11.92%,10.44%
5,,=== 25 features: 35% ===,,,,,
6,Random MR,0.485487,9.782689,7.736432,29.92%,29.4%,28.76%
7,SHAP MR,0.641188,8.169467,6.39218,7.44%,8.06%,6.39%
